![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Labs

----

# Attaching the supply chains to the lineage between systems

The [mapping notebook](mapping-the-systems.ipynb) linked each solution component to the systems that implement it.  That tells Egeria *where* a supply chain runs.  It does not yet tell Egeria *how the data moves* — and that is what lineage relationships between the systems are for.

Two things make this step different from the ones before it.

**Some of the lineage already exists.**  When [Gary Geeke](https://egeria-project.org/practices/coco-pharmaceuticals/personas/gary-geeke/) loaded the Austin and Bucharest inventories, he loaded their system interactions too — one hundred and twenty `DataFlow` relationships, each carrying the label, direction, integration style, protocol, frequency and data exchanged that the site teams had recorded.  Every one of them has an `iscQualifiedName` property still holding the placeholder *"add qualifiedName of information supply chain here"*.  Those relationships are valuable exactly as they are, so this workbook **updates them in place with `mergeUpdate = true`** — the supply chain name is added, and nothing the site teams recorded is lost.

**Lineage relationships are multi-links.**  Most relationship types in Egeria allow only one relationship of a given type between two specific elements.  Lineage types do not: several `DataFlow` relationships may connect the same pair of systems.  So where a hop between two systems carries more than one strategic supply chain — Workday to SAP carries both *Financial Close* and *New Employee Onboarding* — the first chain goes onto the existing relationship, and each further chain gets a **new `DataFlow` cloned from the original**: every property the site team recorded, plus the new `iscQualifiedName` and a label that says which chain it is.  Each chain then has its own lineage, and each can be reported on separately.

Where no interaction was ever loaded — the parent company's own systems have none, and many strategic wires cross hops the site teams did not record — a new `DataFlow` is created from the strategic wire's own description.

----

In [ ]:
# All CSV files live in the data directory
from pathlib import Path

DATA_DIR = Path('./data').resolve()

print(f"Data directory: {DATA_DIR}")
for f in sorted(DATA_DIR.glob('*.csv')):
    print(f"  {f.name}")

In [ ]:
# Helper functions to load CSV files
import csv

def load_csv(filename):
    """Return a list of dicts from a CSV file in DATA_DIR."""
    path = DATA_DIR / filename
    with open(path, newline='', encoding='utf-8') as fh:
        return list(csv.DictReader(fh))

def preview(rows, n=3):
    """Print the first n rows and a row count."""
    print(f"{len(rows)} rows loaded")
    for row in rows[:n]:
        for k, v in row.items():
            print(f"  {k}: {v}")
        print()

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_id = "garygeeke"   # DataFlow relationships anchor on the systems, so this needs write access to every systems zone: Gary owns the inventory
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")
egeria_width = 150

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for user " + user_id)
print("\n")

# These packages support the different types of markdown display
from IPython.display import display, Markdown
from pyegeria import load_mermaid, render_mermaid
load_mermaid()

from datetime import datetime
from collections import defaultdict, Counter
import asyncio
import json
import time


# EgeriaTech combines many of the clients to call Egeria
from pyegeria import EgeriaTech

egeria_client = EgeriaTech(view_server, url, user_id, user_pwd)
token = egeria_client.create_egeria_bearer_token()

----

## The lineage spreadsheet

**`supply-chain-lineage.csv`** has one row for every supply chain that passes over every hop between two systems.  It was derived from the strategic wires in [strategic-supply-chain-analysis.md](../strategic-supply-chain-analysis.md) and the component-to-system mapping: for each wire, in each estate, the systems at either end are looked up, and where that gives an unambiguous pair a row is written for each supply chain the wire implements.  If the pair matches an interaction Gary loaded, the row carries its `interaction_id`; otherwise the row carries the wire's own label, description and data exchanged, ready to create a new relationship from.

Hops where *both* ends map to several systems — a wire between manufacturing execution and the batch record in the parent estate, where each is three per-factory control systems — are not guessed at.  They are listed in **`lineage-needs-decision.csv`** for someone who knows which factory talks to which.

| Column | Meaning |
|---|---|
| `estate` | Coco core, Austin or Bucharest |
| `source_system_qualified_name`, `target_system_qualified_name` | The two systems |
| `interaction_id` | The loaded interaction this hop corresponds to, or blank if none |
| `supply_chain_qualified_name`, `supply_chain_id` | The chain this row attaches |
| `wire_label`, `data_exchanged`, `description` | From the strategic wire — used when a relationship has to be created |

----

In [ ]:
# Load the lineage rows and see what they contain

lineage = load_csv('supply-chain-lineage.csv')
needs_decision = load_csv('lineage-needs-decision.csv')
preview(lineage, 2)

# The direction of each loaded interaction, from the inventory spreadsheets.  It is sent on every update and
# clone because the server-side bean defaults oneWay to true: a merge update that omits it would set it.
INVENTORY_DATA = (Path('..') / 'extending-the-systems-inventory' / 'data').resolve()
direction = {}
for name in ('coco_austin_system_interactions.csv', 'ekg_system_interactions.csv'):
    with open(INVENTORY_DATA / name, newline='', encoding='utf-8') as fh:
        for row in csv.DictReader(fh):
            direction[row['interaction_id']] = (row['direction'] == 'one-way')
print(f"{len(direction)} loaded interactions, {sum(1 for v in direction.values() if not v)} of them two-way")

hops = defaultdict(list)                        # (estate, source, target) -> rows
for r in lineage:
    hops[(r['estate'], r['source_system_qualified_name'], r['target_system_qualified_name'])].append(r)

by_estate = Counter(r['estate'] for r in lineage)
on_existing = sum(1 for r in lineage if r['interaction_id'])
multi = sum(1 for rows in hops.values() if len(rows) > 1)

lines = ["| | Count |", "|---|---|",
         f"| Supply-chain lineage rows | {len(lineage)} |",
         f"| Distinct system hops | {len(hops)} |",
         f"| Hops carrying more than one supply chain | {multi} |",
         f"| Rows on an interaction Gary loaded (update in place, then clone) | {on_existing} |",
         f"| Rows with no loaded interaction (create) | {len(lineage) - on_existing} |"]
for e in ("Coco core", "Austin", "Bucharest"):
    lines.append(f"| … in {e} | {by_estate[e]} |")
lines.append(f"| Wires needing a decision before they can be placed | {len(needs_decision)} |")
display(Markdown("\n".join(lines)))

----

## How each hop is handled

For every hop the workbook resolves the two systems to GUIDs, then:

1. **If the hop is a loaded interaction**, it finds the existing `DataFlow` by its label (the `interaction_id`, which is how the inventory notebook itself finds it) and reads back every property on it.
    * If the relationship's `iscQualifiedName` is still the placeholder, the **first** supply chain is written onto it with `mergeUpdate = true`, together with the flow's direction from the inventory spreadsheet.  (The direction has to be sent every time: the server-side bean defaults `oneWay` to true, so a merge update that omits it would quietly turn a two-way flow into a one-way one.)
    * Every **further** supply chain on the hop becomes a **new** `DataFlow` between the same two systems, carrying all of the original's properties — direction, integration style, protocol, frequency, data exchanged, description — plus the new `iscQualifiedName`.  Its label is the original label with the supply chain id appended, so it can be told apart and so re-running the workbook finds it rather than creating it again.
2. **If nothing was loaded for the hop**, one `DataFlow` is created per supply chain, from the strategic wire's label, description and data exchanged.  The label carries the supply chain id for the same reason.

The workbook is idempotent: it looks for the relationship it is about to create before creating it, and it never overwrites a supply chain name that is already set to something other than the placeholder.  If it finds a relationship it created on an earlier run that is missing its supply chain name, it adds the name rather than creating a duplicate.

----

In [ ]:
# Helpers for reading, updating and creating lineage relationships

PLACEHOLDER = "add qualifiedName of information supply chain here"
VIEW_SERVER = getattr(egeria_client, "view_server", None) or getattr(egeria_client, "server_name", view_server)
LINEAGE_URL = f"{egeria_client.platform_url}/servers/{VIEW_SERVER}/api/open-metadata/lineage-linker"

guid_cache = {}
def guid_of(qualified_name):
    if qualified_name not in guid_cache:
        result = egeria_client.get_element_guid_by_unique_name(qualified_name)
        guid_cache[qualified_name] = None if result == "No elements found" else result
    return guid_cache[qualified_name]

def find_data_flows_by_label(label):
    """Return the DataFlow relationships whose label matches exactly (a list, possibly empty)."""
    result = egeria_client.get_relationships_with_property_value(relationship_type="DataFlow",
                                                                 property_value=label,
                                                                 property_names=["label"])
    return [] if result == "No elements found" else list(result)

def relationship_guid(rel):
    return rel.get('relationshipGUID') or rel.get('guid') or rel.get('relationshipHeader', {}).get('guid')

def relationship_ends(rel):
    e1 = rel.get('elementGUIDAtEnd1') or rel.get('end1', {}).get('guid') or rel.get('elementAtEnd1', {}).get('guid')
    e2 = rel.get('elementGUIDAtEnd2') or rel.get('end2', {}).get('guid') or rel.get('elementAtEnd2', {}).get('guid')
    return e1, e2

def relationship_properties(rel):
    """Flatten the relationship's properties to {name: value}, whatever shape they arrive in."""
    props = rel.get('relationshipProperties') or rel.get('properties') or {}
    if isinstance(props, dict) and 'propertyValueMap' in props:
        flat = {}
        for name, pv in props['propertyValueMap'].items():
            if isinstance(pv, dict):
                flat[name] = pv.get('primitiveValue', pv.get('typedValue', pv.get('value')))
            else:
                flat[name] = pv
        return flat
    if isinstance(props, dict) and 'propertiesAsStrings' in props:
        flat = dict(props['propertiesAsStrings'])
    else:
        flat = {k: v for k, v in props.items() if k not in ('class', 'typeName')}
    if 'oneWay' in flat and isinstance(flat['oneWay'], str):
        flat['oneWay'] = flat['oneWay'].lower() == 'true'
    return flat

def data_flow_properties(properties):
    """Build DataFlowProperties for the lineage linker."""
    return {"class": "DataFlowProperties", **properties}

def update_data_flow(guid, properties):
    """Merge-update a DataFlow: only the properties given are changed, everything else is kept.

    Always include oneWay.  The server-side bean defaults it to true, so a merge update that leaves it
    out would silently set a two-way flow to one-way."""
    assert 'oneWay' in properties, "update_data_flow: oneWay must be supplied explicitly"
    body = {
        "class": "UpdateRelationshipRequestBody",
        "mergeUpdate": True,
        "properties": data_flow_properties(properties)
    }
    # Same pattern as pyegeria's own synchronous methods: run the async request on the kernel's loop.
    # (egeria_client.make_request would wait on run_coroutine_threadsafe() from inside that loop and hang.)
    asyncio.get_event_loop().run_until_complete(
        egeria_client._async_make_request("POST", f"{LINEAGE_URL}/relationships/{guid}/update", body))

def create_data_flow(source_guid, target_guid, properties):
    body = {
        "class": "NewRelationshipRequestBody",
        "properties": data_flow_properties(properties)
    }
    return egeria_client.link_data_flow(element_one_guid=source_guid,
                                        relationship_type_name="DataFlow",
                                        element_two_guid=target_guid,
                                        body=body)


def clone_properties(original_props, new_label, isc_qualified_name):
    """Every property of the original, with a new label and supply chain name."""
    props = {k: v for k, v in original_props.items() if v not in (None, "")}
    props['label'] = new_label
    props['iscQualifiedName'] = isc_qualified_name
    return props

def ensure_properties(rel, wanted):
    """Bring an existing relationship up to the intended property set with a merge update.
    Returns True if anything had to be written.  Never touches a supply chain name that is already
    set to a different chain - that relationship is left alone and reported."""
    have = relationship_properties(rel)
    current_isc = have.get('iscQualifiedName') or ""
    if current_isc not in ("", PLACEHOLDER) and current_isc != wanted.get('iscQualifiedName'):
        return False
    delta = {k: v for k, v in wanted.items() if have.get(k) != v}
    if delta:
        update_data_flow(relationship_guid(rel), delta)
    return bool(delta)

print("Lineage linker endpoint:", LINEAGE_URL)

In [ ]:
# Attach every supply chain to the lineage between the systems

token = egeria_client.create_egeria_bearer_token()

stats = defaultdict(Counter)          # supply chain -> action -> count
unresolved = []

for (estate, source_qn, target_qn), rows in hops.items():
    source_guid, target_guid = guid_of(source_qn), guid_of(target_qn)
    if not source_guid or not target_guid:
        unresolved.append((estate, source_qn if not source_guid else target_qn))
        continue

    chains = []                                      # (supply chain qn, id, row) in spreadsheet order
    for r in rows:
        chains.append((r['supply_chain_qualified_name'], r['supply_chain_id'], r))
    interaction_id = rows[0]['interaction_id']

    original = None
    if interaction_id:
        matches = [m for m in find_data_flows_by_label(interaction_id)
                   if relationship_ends(m) in ((source_guid, target_guid), (None, None))]
        original = matches[0] if matches else None

    if original is not None:
        original_props = relationship_properties(original)
        current_isc = original_props.get('iscQualifiedName') or ""
        remaining = list(chains)

        one_way = direction.get(interaction_id, original_props.get('oneWay', True))
        original_props['oneWay'] = one_way
        if current_isc in ("", PLACEHOLDER):
            isc_qn, isc_id, r = remaining.pop(0)
            update_data_flow(relationship_guid(original), {"iscQualifiedName": isc_qn, "oneWay": one_way})
            stats[isc_qn]['updated in place'] += 1
            print(f"Updated  {interaction_id:<12} {rows[0]['source_system_name']} -> {rows[0]['target_system_name']}  <- {isc_id}")
        else:
            remaining = [c for c in remaining if c[0] != current_isc]
            if len(remaining) < len(chains):
                if ensure_properties(original, {"iscQualifiedName": current_isc, "oneWay": one_way}):
                    stats[current_isc]['existing brought up to date'] += 1
                else:
                    stats[current_isc]['already set'] += 1

        for isc_qn, isc_id, r in remaining:
            new_label = f"{interaction_id} [{isc_id}]"
            present = find_data_flows_by_label(new_label)
            if present:
                repaired = ensure_properties(present[0], clone_properties(original_props, new_label, isc_qn))
                stats[isc_qn]['existing clone brought up to date' if repaired else 'clone already present'] += 1
                continue
            create_data_flow(source_guid, target_guid, clone_properties(original_props, new_label, isc_qn))
            stats[isc_qn]['cloned from interaction'] += 1
            print(f"Cloned   {new_label:<40} {rows[0]['source_system_name']} -> {rows[0]['target_system_name']}")
    else:
        for isc_qn, isc_id, r in chains:
            new_label = f"{isc_id}: {r['wire_label']}"
            existing = [m for m in find_data_flows_by_label(new_label)
                        if relationship_ends(m) in ((source_guid, target_guid), (None, None))]
            wanted = {"iscQualifiedName": isc_qn, "label": new_label, "description": r['description'],
                      "dataExchanged": r['data_exchanged'], "oneWay": True}
            if existing:
                stats[isc_qn]['existing brought up to date' if ensure_properties(existing[0], wanted) else 'already present'] += 1
                continue
            create_data_flow(source_guid, target_guid, wanted)
            stats[isc_qn]['created from wire'] += 1
            print(f"Created  {new_label:<55} {r['source_system_name']} -> {r['target_system_name']}  [{estate}]")

print(f"\nDone - {sum(sum(c.values()) for c in stats.values())} relationships processed across {len(hops)} hops.")
if unresolved:
    print(f"\n{len(unresolved)} hops skipped because a system is not in Egeria - has the inventory notebook been run?")
    for e, qn in sorted(set(unresolved)): print(f"   [{e}] {qn}")

----

## What was done, by supply chain

Each supply chain now has its own lineage between systems: relationships updated in place where Gary's interaction was the only chain on the hop, cloned where a hop carries several, created where nothing had been loaded.  The wires that could not be placed without a decision are listed separately — they are not gaps in the lineage, they are questions for the people who run the factories.

----

In [ ]:
# Report by supply chain, and the decisions still needed

ACTIONS = ['updated in place', 'cloned from interaction', 'created from wire', 'existing brought up to date', 'existing clone brought up to date', 'already set', 'already present', 'clone already present']
out = ["# Supply chain lineage report", "",
       f"Generated {datetime.now():%Y-%m-%d %H:%M} from `supply-chain-lineage.csv` - {len(lineage)} rows over {len(hops)} system hops.", "",
       "## By supply chain", "",
       "| Supply chain | " + " | ".join(ACTIONS) + " |",
       "|---|" + "---|" * len(ACTIONS)]
for isc_qn in sorted(stats):
    short = isc_qn.replace("InformationSupplyChain::", "").replace(" Information Supply Chain", "")
    out.append(f"| {short} | " + " | ".join(str(stats[isc_qn][a]) for a in ACTIONS) + " |")
out += ["", "## By estate", "", "| Estate | Hops | Rows |", "|---|---|---|"]
for e in ("Coco core", "Austin", "Bucharest"):
    out.append(f"| {e} | {sum(1 for k in hops if k[0]==e)} | {by_estate[e]} |")
out += ["", "## Wires that need a decision before they can be placed", "",
        "Both ends map to several systems in the estate. Someone who knows which system talks to which needs to pick the pair(s); "
        "add the chosen rows to `supply-chain-lineage.csv` and re-run.", "",
        "| Estate | From component | To component | Wire | Systems at each end |", "|---|---|---|---|---|"]
for d in needs_decision:
    out.append(f"| {d['estate']} | {d['component1']} | {d['component2']} | {d['wire_label']} | {d['systems_at_component1']} → {d['systems_at_component2']} |")
report = "\n".join(out)
display(Markdown(report))
report_path = Path('./supply-chain-lineage-report.md').resolve()
report_path.write_text(report, encoding='utf-8')
print(f"Report written to {report_path}")

----

## Checking one hop

Workday to SAP at Austin is the hop that shows the whole mechanism: Gary's original interaction, now carrying *Financial Close and External Reporting*, and beside it a clone carrying *New Employee Onboarding* with every other property identical.

----

In [ ]:
# Show every DataFlow on one hop, with the supply chain each one carries

token = egeria_client.create_egeria_bearer_token()

sample = next((r for r in lineage if r['estate'] == 'Austin' and r['interaction_id']
               and 'Workday' in r['source_system_name'] and 'SAP' in r['target_system_name']), lineage[0])
labels = [sample['interaction_id']] if sample['interaction_id'] else []
labels += [f"{sample['interaction_id']} [{r['supply_chain_id']}]" if sample['interaction_id'] else f"{r['supply_chain_id']}: {r['wire_label']}"
           for r in hops[(sample['estate'], sample['source_system_qualified_name'], sample['target_system_qualified_name'])]]

print(f"{sample['source_system_name']} -> {sample['target_system_name']}  [{sample['estate']}]\n")
for label in dict.fromkeys(labels):
    for rel in find_data_flows_by_label(label):
        props = relationship_properties(rel)
        print(f"  {props.get('label','?'):<45} isc = {props.get('iscQualifiedName','')}")
        print(f"      {', '.join(f'{k}={v}' for k, v in props.items() if k not in ('label','iscQualifiedName','description'))}")

----

## What this makes possible

Each strategic supply chain is now connected end to end: from the chain, through its solution components, through the `ImplementedBy` links to the systems, and along the `DataFlow` relationships between those systems — with every relationship naming the chain it belongs to.  Selecting a supply chain in Egeria Explorer shows the implementation graph assembled from all of that, which is what turns a designed supply chain into a monitored one.

The site teams' interaction data survived intact.  Nothing was replaced; the supply chain names were merged in, and where a hop carries several chains the extra relationships carry the site team's properties forward unchanged.

----